# Plafond avec les indices seulement

Question : **si on connaissait la date de chaque ligne**, quelle accuracy obtiendrait-on en utilisant seulement les rendements des indices du lendemain ?

- dates : `calendar/train_dates.csv` (on garde le morceau daté par les jours fériés, 2007-2015, le plus fiable)
- on apprend sur 2007-2012, on mesure sur 2013-2015
- la target d'une ligne datée d = rendement de la séance suivante de la bourse de son groupe

⚠️ Ce n'est pas une prévision : on utilise les rendements réels du jour de la target. C'est juste pour savoir jusqu'où cette piste peut aller.

In [1]:
import numpy as np
import pandas as pd
import exchange_calendars as xcals
import warnings
warnings.filterwarnings("ignore")

X = pd.read_csv("data/X_train.csv", index_col="ROW_ID")
y = pd.read_csv("data/y_train.csv", index_col="ROW_ID")["target"]
dates = pd.read_csv("calendar/train_dates.csv", parse_dates=["date"])
index_returns = pd.read_csv("market_data/returns.csv", index_col="Date", parse_dates=True)

# only the dates found with the holidays (most reliable)
dates = dates[(dates["method"] == "holidays") & dates["date"].notna()]
X["date"] = X["TS"].map(dict(zip(dates["TS"], dates["date"])))
X["target"] = y
X = X.dropna(subset=["date"])
print(len(X), "rows,", X["date"].min().date(), "->", X["date"].max().date())

498409 rows, 2007-09-26 -> 2015-10-30


## La séance de la target (= séance suivante de la bourse du groupe)

In [2]:
group_exchange = {1: "XTKS", 2: "XHKG", 3: "XKRX", 4: "XSHG"}
own_index = {1: "NIKKEI225", 2: "HSCEI", 3: "KOSPI", 4: "SHANGHAI"}

X["next_date"] = pd.NaT
for g, code in group_exchange.items():
    sessions = pd.DatetimeIndex(xcals.get_calendar(code, start="2004-01-01").sessions)
    mask = X["GROUP"] == g
    pos = sessions.searchsorted(X.loc[mask, "date"], side="right")
    X.loc[mask, "next_date"] = sessions[pos]

X[["TS", "GROUP", "date", "next_date"]].head()

,TS,GROUP,date,next_date
ROW_ID,,,,
0,DATE_0001,1,2013-07-26,2013-07-29
1,DATE_0001,4,2013-07-26,2013-07-29
2,DATE_0001,1,2013-07-26,2013-07-29
3,DATE_0001,2,2013-07-26,2013-07-29
4,DATE_0001,4,2013-07-26,2013-07-29


## Les variables

- rendement des 10 indices le jour de la target
- rendement des indices occidentaux le jour d (ils clôturent après l'Asie, avant la séance de la target)

In [3]:
western = ["SP500", "DAX", "CAC40", "FTSE100", "SX5E"]

next_ret = index_returns.reindex(X["next_date"]).fillna(0)
next_ret.columns = [c + "_next" for c in next_ret.columns]
same_west = index_returns[western].reindex(X["date"]).fillna(0)
same_west.columns = [c + "_same_day" for c in same_west.columns]

F = pd.concat([next_ret.reset_index(drop=True), same_west.reset_index(drop=True)], axis=1)
F.index = X.index
F["const"] = 1.0

fit = X["date"] < "2013-01-01"
hold = ~fit
y_hold = (X.loc[hold, "target"] > 0).astype(int)
print("fit rows:", fit.sum(), "| held-out rows:", hold.sum())

fit rows: 307657 | held-out rows: 190752


## Points de comparaison

In [4]:
results = {}
results["always 1"] = (y_hold == 1).mean()
results["sign(RET_1)"] = ((X.loc[hold, "RET_1"] > 0).astype(int) == y_hold).mean()
results

{'always 1': np.float64(0.5015517530615669),
 'sign(RET_1)': np.float64(0.5146001090421071)}

## A. Un seul chiffre par groupe : le signe de l'indice de sa bourse le lendemain

In [5]:
pred_a = pd.Series(np.nan, index=X.index)
for g, name in own_index.items():
    m_fit = fit & (X["GROUP"] == g)
    m_hold = hold & (X["GROUP"] == g)
    # does the group go with or against its index ? learned on 2007-2012
    sign = np.sign(np.corrcoef(X.loc[m_fit, "target"], F.loc[m_fit, name + "_next"])[0, 1])
    pred_a[m_hold] = sign * F.loc[m_hold, name + "_next"]
    print(f"group {g} ({name}): sign {sign:+.0f}")

results["A. group sign x own index"] = ((pred_a[hold] > 0).astype(int) == y_hold).mean()

group 1 (NIKKEI225): sign -1
group 2 (HSCEI): sign -1
group 3 (KOSPI): sign -1
group 4 (SHANGHAI): sign -1


## B. Une régression par allocation, indice de sa bourse seulement

In [6]:
def fit_per_allocation(columns, ridge=1e-6):
    pred = pd.Series(np.nan, index=X.index)
    for alloc, sub in X.groupby("ALLOCATION"):
        rows_fit = sub.index[fit[sub.index]]
        rows_hold = sub.index[hold[sub.index]]
        if len(rows_fit) < 100 or len(rows_hold) == 0:
            continue
        A = F.loc[rows_fit, columns].values
        b = X.loc[rows_fit, "target"].values
        beta = np.linalg.solve(A.T @ A + ridge * len(b) * np.eye(A.shape[1]), A.T @ b)
        pred[rows_hold] = F.loc[rows_hold, columns].values @ beta
    return pred

pred_b = pd.Series(np.nan, index=X.index)
for g, name in own_index.items():
    p = fit_per_allocation([name + "_next", "const"])
    in_group = X["GROUP"] == g
    pred_b[in_group] = p[in_group]

ok = pred_b[hold].notna()
results["B. per allocation, own index"] = ((pred_b[hold][ok] > 0).astype(int) == y_hold[ok]).mean()

## C. Une régression par allocation, tous les indices

In [7]:
pred_c = fit_per_allocation(list(F.columns))
ok = pred_c[hold].notna()
results["C. per allocation, all indices"] = ((pred_c[hold][ok] > 0).astype(int) == y_hold[ok]).mean()

pd.Series(results).round(4).to_frame("accuracy 2013-2015")

,accuracy 2013-2015
always 1,0.5016
sign(RET_1),0.5146
A. group sign x own index,0.5410
"B. per allocation, own index",0.5470
"C. per allocation, all indices",0.5373


## Détail par groupe et selon la confiance

In [8]:
table = pd.DataFrame({
    "group": X.loc[hold, "GROUP"],
    "correct_A": (pred_a[hold] > 0).astype(int) == y_hold,
    "correct_C": (pred_c[hold] > 0).astype(int) == y_hold,
})
print(table.groupby("group")[["correct_A", "correct_C"]].mean().round(3))

# when the predicted move is big, is it more often right ?
strength = pred_c[hold].abs()
for q in [0.0, 0.5, 0.8]:
    m = strength >= strength.quantile(q)
    print(f"top {100 - int(q * 100)}% strongest predictions: {table.loc[m, 'correct_C'].mean():.1%}")

       correct_A  correct_C
group                      
1          0.556      0.544
2          0.555      0.551
3          0.506      0.507
4          0.549      0.548
top 100% strongest predictions: 53.7%
top 50% strongest predictions: 56.6%
top 20% strongest predictions: 61.7%


## Et si la date est fausse ?

Sur le test, on ne trouve la bonne date qu'environ 1 fois sur 3. Simulation : on décale la date d'une partie des lignes (n'importe quel autre jour) et on regarde ce qu'il reste.

In [9]:
rng = np.random.default_rng(0)
correct_c = table["correct_C"].values
for share_right in [1.0, 0.6, 0.3]:
    right = rng.random(len(correct_c)) < share_right
    # with a wrong date the prediction is a coin flip
    sim = np.where(right, correct_c, rng.random(len(correct_c)) < 0.5)
    print(f"dates right {share_right:.0%} -> accuracy ~ {sim.mean():.1%}")

dates right 100% -> accuracy ~ 53.7%
dates right 60% -> accuracy ~ 52.2%
dates right 30% -> accuracy ~ 51.0%
